# Notebook 03: Evaluation
1. **Clean accuracy:** each full pipeline on the GSM-Symbolic sample, Exact Match (trained adapters, and the
   untrained base model with the same prompts as a baseline)
2. **Known-correct intersection:** trap candidates from notebook 01 whose problem all three conditions solve
3. **Manual review** of the first `TRAP_TEST_SIZE` (100) traps in the intersection
4. **Trap Test:** every condition answers from the same clean and trapped traces, at several training
   checkpoints
5. **Analysis:** trace dependence (TD), answer changes, Other Errors, Cochran's Q, McNemar + Holm

Decoding is greedy (no sampling) everywhere. Every generation is appended to a JSONL file as it is made,
so any stage can be interrupted and resumed by re-running its cell.

**Modular stage isolation.** Stages run one after another over the whole question set: the Thinker
adapter generates every trace and is unloaded, then each Solver adapter is loaded and run. Each
`generate()` call starts a new KV cache from its own input, and Solver inputs are rebuilt from text,
so the Hard Bottleneck Solver's input contains only R (see `src/faithfulness/inference.py`).

In [ ]:
import sys
sys.path.insert(0, "../src")

from faithfulness import config

import pandas as pd
from faithfulness.analysis import CONDITIONS, exact_match_table, paired_tests, td_by_step, trap_summary
from faithfulness.data import auto_check_trap, load_json, load_jsonl, save_json
from faithfulness.evaluation import run_clean_evaluation, run_trap_test

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
problems = load_json(config.GSM_SYMBOLIC_PATH)
print(f"{len(problems)} GSM-Symbolic questions")

## 1. Clean mathematical accuracy
Each condition runs its full pipeline: Monolithic writes trace and answer in one generation; the modular
conditions share one Thinker pass, then each Solver answers. A Thinker trace that does not close `</think>`
within `MAX_NEW_TOKENS_TRACE` counts as wrong for both modular conditions.

Each question needs two reasoning-length generations plus two Solver passes, so on a 4070 the 1,500-question
sample takes about a day and the full 5,000 (`SYMBOLIC_PER_TEMPLATE = None`) takes days. Lower `config.GEN_BATCH_SIZE` if you run out of memory.

In [ ]:
clean = run_clean_evaluation(problems, step="final")
clean.to_csv(config.EVAL_DIR / "clean_results_final.csv", index=False)

em = exact_match_table(clean)
em.to_csv(config.RESULTS_DIR / "clean_exact_match.csv")
em

In [ ]:
# Baseline: the untrained base model in the same three pipelines (same prompts, no adapters)
RUN_UNTRAINED_BASELINE = True

if RUN_UNTRAINED_BASELINE:
    clean_base = run_clean_evaluation(problems, step=0)
    clean_base.to_csv(config.EVAL_DIR / "clean_results_0.csv", index=False)
    em_both = pd.concat({"untrained": exact_match_table(clean_base), "trained": exact_match_table(clean)}, axis=1)
    em_both.to_csv(config.RESULTS_DIR / "clean_exact_match_vs_untrained.csv")
    display(em_both)

In [ ]:
# Unfinished traces (no </think> within the limit; mostly reasoning loops) and unparseable answers
clean.groupby("condition").agg(
    trace_unfinished=("trace_unfinished", "sum"),
    no_answer_parsed=("answer", lambda s: (s == "").sum()),
).reindex(CONDITIONS)

## 2. Known-correct intersection
Only problems that **all three** trained conditions answer correctly before any intervention are eligible, so
a trap-induced error cannot be confused with a problem a model could not solve. The traps themselves were
written in notebook 01 (from the base model's own correct trace); here they are kept in their seeded,
template-balanced selection order if their problem is in the intersection and the trap passes the automatic
checks.

In [ ]:
wide = clean.pivot(index="id", columns="condition", values="correct")[CONDITIONS]
intersection = set(wide.index[wide.all(axis=1)])
print(f"Correct in all three conditions: {len(intersection)} / {len(wide)}")

selection = load_json(config.TRAP_DIR / "trap_selection.json")
traps = {r["id"]: r for r in load_jsonl(config.TRAP_DIR / "trap_candidates.jsonl") if not r["error"]}
auto_problems = {pid: auto_check_trap(traps[pid]) for pid in selection if pid in traps}
eligible = [pid for pid in selection if pid in intersection and pid in auto_problems and not auto_problems[pid]]

print(f"Trap candidates: {len(selection)} | generated: {len(auto_problems)} | "
      f"in the intersection: {sum(pid in intersection for pid in auto_problems)} | "
      f"also passed automatic checks: {len(eligible)}")
print(f"Templates among eligible traps: {len({traps[pid]['template_id'] for pid in eligible})}")

## 3. Manual review
Each trap in the main Trap Test set is checked by hand. For each, confirm that:
1. the trace contains an error,
2. later steps follow from that error, and
3. the answer implied by the trace differs from the correct answer (and equals `trap_answer`).

The next cell writes `data/trap/trap_review.csv`: eligible traps in selection order, until there are
`TRAP_TEST_SIZE` (100) rows not marked `no`. Open it, set `approved` to `yes` or `no` for each row (and add
`notes` if useful), save it, and re-run the cell: each rejection pulls in the next trap. Repeat until every row
is decided. Re-running never overwrites decisions already made. The cell after it prints each changed step and
a diff of the two traces to help.

In [ ]:
review_path = config.TRAP_DIR / "trap_review.csv"
existing = pd.read_csv(review_path, dtype=str).fillna("") if review_path.exists() else pd.DataFrame()
decided = dict(zip(existing.get("id", []), zip(existing.get("approved", []), existing.get("notes", []))))

# Walk the selection order until TRAP_TEST_SIZE rows are approved or still undecided
rows, kept = [], 0
for pid in eligible:
    if kept == config.TRAP_TEST_SIZE:
        break
    approved, notes = decided.get(pid, ("", ""))
    kept += approved.lower() != "no"
    rows.append({
        "order": len(rows), "id": pid, "template_id": traps[pid]["template_id"],
        "correct_answer": traps[pid]["target_answer"], "trap_answer": traps[pid]["trap_answer"],
        "original_step": traps[pid]["original_step"], "modified_step": traps[pid]["modified_step"],
        "approved": approved, "notes": notes,
    })
review = pd.DataFrame(rows)
review.to_csv(review_path, index=False)

print(f"{len(review)} traps written to {review_path}")
print(f"Approved: {(review['approved'].str.lower() == 'yes').sum()} | "
      f"rejected: {(review['approved'].str.lower() == 'no').sum()} | undecided: {(review['approved'] == '').sum()}")

In [ ]:
import difflib

def show_trap(pid):
    t = traps[pid]
    print(f"{'=' * 80}\n{pid} | correct {t['target_answer']} -> trap {t['trap_answer']} | "
          f"auto-check: {'; '.join(auto_check_trap(t)) or 'ok'}")
    print(f"CHANGED STEP\n  - {t['original_step']}\n  + {t['modified_step']}\nDIFF")
    for line in difflib.unified_diff(t["clean_trace"].splitlines(), t["trapped_trace"].splitlines(), lineterm="", n=0):
        if not line.startswith(("---", "+++", "@@")):
            print("  " + line)

# Show the traps that still need a decision (adjust the slice to page through them)
pending = review.loc[review["approved"] == "", "id"].tolist()
for pid in pending[:10]:
    show_trap(pid)

In [ ]:
review = pd.read_csv(review_path, dtype=str).fillna("")
review = review.sort_values("order", key=lambda s: s.astype(int))
approved_ids = review.loc[review["approved"].str.lower() == "yes", "id"].tolist()[:config.TRAP_TEST_SIZE]
n_rejected = int((review["approved"].str.lower() == "no").sum())

# Main set: approved by hand. Secondary set: every eligible trap (automatic checks only).
trap_set = [traps[pid] for pid in approved_ids]
trap_set_all = [traps[pid] for pid in eligible]
save_json(trap_set, config.TRAP_DIR / "trap_set.json")
save_json(trap_set_all, config.TRAP_DIR / "trap_set_all.json")
print(f"Main trap set: {len(trap_set)} problems (reviewed {len(approved_ids) + n_rejected}, rejected {n_rejected}, "
      f"undecided {(review['approved'] == '').sum()})")
print(f"Secondary trap set (automatic checks only): {len(trap_set_all)} problems")
if len(trap_set) < config.TRAP_TEST_SIZE:
    print("Fewer than TRAP_TEST_SIZE approved: re-run the review cell and decide the new rows")

## 4. Trap Test
Every condition receives the same trace, in the input format of its answer stage:

| Condition | Input | Generates |
| :--- | :--- | :--- |
| Monolithic | $Q$ + `<think>` $R$ `</think>` as a forced prefix | $A$ |
| Soft Modular | $(Q, R)$ to the Soft Solver | $A$ |
| Hard Bottleneck | $R$ to the Hard Solver | $A$ |

Each problem is run twice per condition: with the unmodified trace $R$ and with $R_{trap}$.

The main set is tested at every step in `config.CHECKPOINT_STEPS` (0 = the untrained base model with the same
prompts). The secondary set is tested on the untrained model and the final adapters. Both write to the same
files, so problems in both sets are generated once.

In [ ]:
SECONDARY_STEPS = [0, "final"]

trap_runs = []
for step in config.CHECKPOINT_STEPS:
    trap_runs.append(run_trap_test(trap_set, step=step))
for step in SECONDARY_STEPS:
    trap_runs.append(run_trap_test(trap_set_all, step=step))

trap_all = pd.concat(trap_runs).drop_duplicates(["id", "condition", "step"])
trap_all.to_csv(config.TRAP_DIR / "trap_results.csv", index=False)

main_ids = {t["id"] for t in trap_set}
trap = trap_all[trap_all["id"].isin(main_ids) & (trap_all["step"] == "final")]
print(f"Main set, final adapters: {trap['id'].nunique()} problems x {trap['condition'].nunique()} conditions")

## 5. Analysis
- **TD** = trap follows / trap tests. A follow means the answer matches the answer implied by $R_{trap}$.
- **Other errors**: neither the correct answer nor the trap answer; never counted as a follow.
- **Answer changed**: the answer under $R_{trap}$ differs from the answer under the unmodified $R$.
- **Clean trace correct**: with the unmodified supplied trace, the answer is still correct (a check that the
  input format itself does not break the condition).

Following a trapped trace is evidence of answer–trace agreement, not proof that the trace represents the
model's internal computation.

In [ ]:
td = trap_summary(trap)
td.to_csv(config.RESULTS_DIR / "trap_dependence.csv")
td

### Paired tests
The same problems are used in every condition, so outcomes are paired. Cochran's Q tests whether follow
rates differ across the three conditions; if it does, exact McNemar tests compare each pair, with Holm
correction across the three comparisons. Soft Modular vs. Hard Bottleneck is the key comparison for the
information bottleneck hypothesis (both two-stage; only question access differs).

In [ ]:
omnibus, pairwise = paired_tests(trap)
print(f"Cochran's Q = {omnibus['Q']:.3f}, df = {omnibus['df']}, p = {omnibus['p_value']:.4g} "
      f"(n = {omnibus['n_problems']} problems)")
if omnibus["p_value"] >= 0.05:
    print("Omnibus test not significant: pairwise tests are shown for completeness only")
pairwise

### Trace dependence through training
TD on the main set at each checkpoint. Step 0 is the untrained base model given the same prompts, which shows
how much of each condition's trace dependence comes from the input format alone and how much from training.

In [ ]:
td_steps = td_by_step(trap_all[trap_all["id"].isin(main_ids)])
td_steps.to_csv(config.RESULTS_DIR / "trap_dependence_by_step.csv")
td_steps.round(3)

### Secondary set (automatic checks only)
The same tests on every eligible trap, for the untrained model and the final adapters. These traps were not
checked by hand, so this is reported as a robustness check with more statistical power, not as the main result.

In [ ]:
all_ids = {t["id"] for t in trap_set_all}
secondary = {}
for step in SECONDARY_STEPS:
    sub = trap_all[trap_all["id"].isin(all_ids) & (trap_all["step"] == step)]
    secondary[step] = (trap_summary(sub), *paired_tests(sub))
    print(f"===== step {step}: {sub['id'].nunique()} problems =====")
    display(secondary[step][0])
    om = secondary[step][1]
    print(f"Cochran's Q = {om['Q']:.3f}, p = {om['p_value']:.4g}")
    display(secondary[step][2])

In [ ]:
results = {
    "clean_exact_match": em.reset_index().to_dict(orient="records"),
    "trap_dependence": td.reset_index().to_dict(orient="records"),
    "cochrans_q": omnibus,
    "mcnemar_holm": pairwise.to_dict(orient="records"),
    "trap_dependence_by_step": td_steps.reset_index().astype({"step": str}).to_dict(orient="records"),
    "secondary_set": {str(step): {"trap_dependence": s[0].reset_index().to_dict(orient="records"),
                                  "cochrans_q": s[1], "mcnemar_holm": s[2].to_dict(orient="records")}
                      for step, s in secondary.items()},
    "trap_selection": {
        "candidates": len(selection),
        "traps_generated": len(auto_problems),
        "intersection_size": len(intersection),
        "eligible": len(eligible),
        "manually_reviewed": len(approved_ids) + n_rejected,
        "manually_rejected": n_rejected,
        "trap_test_size": len(trap_set),
    },
    "settings": {
        "base_model": config.BASE_MODEL,
        "max_new_tokens_trace": config.MAX_NEW_TOKENS_TRACE,
        "max_new_tokens_answer": config.MAX_NEW_TOKENS_ANSWER,
        "checkpoint_steps": [str(s) for s in config.CHECKPOINT_STEPS],
        "decoding": "greedy (do_sample=False)",
        "generation_batch_size": config.GEN_BATCH_SIZE,
    },
}
save_json(results, config.RESULTS_DIR / "results.json")
pairwise.to_csv(config.RESULTS_DIR / "mcnemar_holm.csv", index=False)
print(f"Saved results to {config.RESULTS_DIR}")